# Verify the prepared Landsat surface-temperature layer

This optional notebook reads the same cached 30 m analytical rasters as the local map. The production pipeline remains in `greenwave_local_layers.landsat`; this notebook performs no downloads or package installation.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
from greenwave_local_layers.landsat import open_cached_observation

project_root = Path.cwd()
if project_root.name == 'playground':
    project_root = project_root.parent
manifest_path = project_root / '.cache' / 'local-layers' / 'landsat-temperature' / 'manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
[(item['value'], item['kind']) for item in manifest['timelineItems']]

Choose a prepared observation. Status 1 is clear, 2 is cloud-obscured and 0 is other missing data. Temperatures beneath clouds are never estimated.

In [ ]:
observation_id = manifest['defaultObservation']
temperature_c, status, uncertainty_k, grid = open_cached_observation(observation_id)
clear = status == 1
print(observation_id, grid)
print(f'Clear pixels: {clear.mean():.1%}')
print(f'Median clear-sky surface temperature: {np.nanmedian(temperature_c[clear]):.2f} °C')

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
image = axes[0].imshow(np.where(clear, temperature_c, np.nan), cmap='inferno', vmin=15, vmax=50)
axes[0].set_title(f'{observation_id}: clear-sky surface temperature')
axes[0].set_axis_off()
figure.colorbar(image, ax=axes[0], label='°C', shrink=0.8)
axes[1].imshow(status, cmap='tab10', vmin=0, vmax=9)
axes[1].set_title('QA status: 0 missing, 1 clear, 2 cloud')
axes[1].set_axis_off()
plt.show()

The fixed 15-50 °C scale matches the map. Land-surface temperature is not air temperature and describes conditions around one satellite overpass.